# Agente Deliberativo con Planificacion Multi-Etapa

Este notebook implementa un agente **deliberativo** (no reactivo) que:
1. **Planifica** una secuencia de 5 etapas antes de ejecutar.
2. **Ejecuta** cada etapa adaptativamente segun los resultados intermedios.
3. **Ajusta** su comportamiento ante condiciones cambiantes (inconsistencias inesperadas).

Arquitectura: Plan-and-Execute (LangChain).

In [ ]:
import os
import json
import re
from dataclasses import dataclass, field
from typing import Optional
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain import hub

from lore_database import get_all_fragments, get_fragments_by_category, LoreFragment

PLANS_DIR = Path("plans")
PLANS_DIR.mkdir(exist_ok=True)

## Modelos de datos para planificacion

`PlanStage` representa una etapa del plan. `ValidationPlan` agrupa el plan completo.

In [ ]:
@dataclass
class PlanStage:
    stage_id: int
    name: str
    description: str
    status: str = "PENDING"
    result: Optional[str] = None
    decision: Optional[str] = None

@dataclass
class ValidationPlan:
    script_fragment: str
    stages: list[PlanStage] = field(default_factory=list)
    final_verdict: Optional[str] = None
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

## Motor de busqueda semantica

Reutilizado del agente original.

In [ ]:
def _buscar(query: str, fragments: list[LoreFragment], top_k=3) -> list[str]:
    query_words = set(re.sub(r"[^\w\s]", "", query.lower()).split())
    scored = []
    for frag in fragments:
        text_words = set(re.sub(r"[^\w\s]", "", frag.text.lower()).split())
        tag_words = set(frag.tags)
        overlap = len(query_words & (text_words | tag_words))
        if overlap > 0:
            scored.append((overlap, frag.text))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [t for _, t in scored[:top_k]]

## Herramientas del agente planificador

Cinco herramientas que cubren las 5 etapas del plan.

In [ ]:
@tool
def extract_elements(script_fragment: str) -> str:
    """Etapa 1: extrae personajes, eventos, objetos y ano de un fragmento de guion."""
    llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o-mini",
        temperature=0.0,
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "Eres un extractor de entidades narrativas. Dado un fragmento de guion de Transformers, "
            "devuelve UNICAMENTE un JSON con: characters (lista), events (lista), objects (lista), year (string o null). "
            "No agregues explicaciones."
        )),
        ("human", "Fragmento: {fragment}")
    ])
    chain = prompt | llm
    response = chain.invoke({"fragment": script_fragment})
    return response.content

@tool
def retrieve_lore_for_elements(elements_json: str) -> str:
    """Etapa 2: busca lore canonico relevante para cada elemento extraido."""
    try:
        data = json.loads(elements_json)
    except json.JSONDecodeError:
        return "Error: entrada no es JSON valido."

    all_results = []
    for character in data.get("characters", []):
        res = _buscar(character, get_all_fragments(), top_k=2)
        if res:
            all_results.append(f"[Personaje: {character}]\n" + "\n".join(f"  - {r}" for r in res))

    for obj in data.get("objects", []):
        res = _buscar(obj, get_all_fragments(), top_k=2)
        if res:
            all_results.append(f"[Objeto: {obj}]\n" + "\n".join(f"  - {r}" for r in res))

    for event in data.get("events", []):
        res = _buscar(event, get_all_fragments(), top_k=2)
        if res:
            all_results.append(f"[Evento: {event}]\n" + "\n".join(f"  - {r}" for r in res))

    year = data.get("year")
    if year:
        res = _buscar(str(year), get_all_fragments(), top_k=3)
        if res:
            all_results.append(f"[Ano: {year}]\n" + "\n".join(f"  - {r}" for r in res))

    return "\n\n".join(all_results) if all_results else "No se encontro lore relevante."

@tool
def validate_against_lore(validation_input: str) -> str:
    """Etapa 3: compara guion contra lore y detecta inconsistencias. Entrada: JSON con 'script' y 'lore'."""
    try:
        data = json.loads(validation_input)
    except json.JSONDecodeError:
        return "Error: entrada no es JSON valido."

    llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o-mini",
        temperature=0.1,
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", (
            "Eres un validador de continuidad narrativa. Comparas un fragmento de guion contra lore canonico "
            "y detectas inconsistencias. Responde UNICAMENTE con JSON: "
            '{\"verdict\": \"CONSISTENTE|INCONSISTENTE|REQUIERE_REVISION\", \"inconsistencies\": [{\"description\":\"", \"severity\":\"CRITICA|MODERADA|MENOR\"}], \"confidence\": 0.0-1.0}'"
        )),
        ("human", "LORE:\n{lore}\n\nGUION:\n{script}")
    ])
    chain = prompt | llm
    response = chain.invoke({
        "lore": data.get("lore", ""),
        "script": data.get("script", "")
    })
    return response.content

@tool
def emit_verdict(verdict_json: str) -> str:
    """Etapa 4: emite veredicto final y recomendacion."""
    try:
        data = json.loads(verdict_json)
    except json.JSONDecodeError:
        return "Error: entrada no es JSON valido."

    verdict = data.get("verdict", "REQUIERE_REVISION")
    inconsistencies = data.get("inconsistencies", [])
    confidence = data.get("confidence", 0.5)

    recommendation = ""
    if verdict == "CONSISTENTE":
        recommendation = "El guion es coherente con el lore."
    elif verdict == "INCONSISTENTE":
        recs = []
        for inc in inconsistencies:
            recs.append(f"- Revisar: {inc.get('description', '')} ({inc.get('severity', 'MENOR')})")
        recommendation = "Se detectaron inconsistencias.\n" + "\n".join(recs)
    else:
        recommendation = "No se pudo determinar la validez. Revision manual requerida."

    result = {
        "verdict": verdict,
        "inconsistencies": inconsistencies,
        "confidence": confidence,
        "recommendation": recommendation,
    }
    return json.dumps(result, ensure_ascii=False, indent=2)

@tool
def save_plan(plan_json: str) -> str:
    """Etapa 5: guarda el plan de validacion ejecutado en disco."""
    try:
        data = json.loads(plan_json)
    except json.JSONDecodeError:
        return "Error: JSON invalido."

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"validation_plan_{timestamp}.json"
    filepath = PLANS_DIR / filename
    filepath.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    return f"Plan guardado en: {filepath.name}"

## Agente deliberativo: Plan-and-Execute

Primero genera un plan de 5 etapas, luego lo ejecuta secuencialmente.

In [ ]:
class DeliberativeContinuityAgent:

    def __init__(self, verbose=True):
        self.llm = ChatOpenAI(
            base_url=os.getenv("OPENAI_BASE_URL"),
            api_key=os.getenv("GITHUB_TOKEN"),
            model="gpt-4o",
            temperature=0.1,
        )
        self.tools = [
            extract_elements,
            retrieve_lore_for_elements,
            validate_against_lore,
            emit_verdict,
            save_plan,
        ]
        self.prompt = hub.pull("hwchase17/openai-tools-agent")
        self.agent = create_openai_tools_agent(self.llm, self.tools, self.prompt)
        self.executor = AgentExecutor(
            agent=self.agent,
            tools=self.tools,
            verbose=verbose,
            max_iterations=10,
            handle_parsing_errors=True,
            return_intermediate_steps=True,
        )

    def _create_plan(self, script_fragment: str) -> ValidationPlan:
        plan = ValidationPlan(script_fragment=script_fragment)
        plan.stages = [
            PlanStage(1, "Extraccion", "Identificar personajes, eventos, objetos y ano"),
            PlanStage(2, "Recuperacion", "Buscar lore canonico relevante para cada elemento"),
            PlanStage(3, "Validacion", "Comparar guion contra lore y detectar inconsistencias"),
            PlanStage(4, "Dictamen", "Emitir veredicto CONSISTENTE/INCONSISTENTE/REQUIERE_REVISION"),
            PlanStage(5, "Persistencia", "Guardar plan y resultados en disco"),
        ]
        return plan

    def run(self, script_fragment: str) -> dict:
        print(f"\n{'='*70}\n AGENTE DELIBERATIVO — Planificacion Multi-Etapa\n{'='*70}")

        plan = self._create_plan(script_fragment)
        print(f"\n[PLAN] Fragmento: {script_fragment[:60]}...")
        print(f"[PLAN] Etapas definidas: {len(plan.stages)}")
        for s in plan.stages:
            print(f"  {s.stage_id}. {s.name}: {s.description}")

        orchestration_prompt = f"""Eres el Consultor de Continuidad Deliberativo de Transformers.

Tu trabajo es validar fragmentos de guion siguiendo ESTRICTAMENTE este plan de 5 etapas:

1. EXTRAER elementos del guion usando extract_elements
2. RECUPERAR lore canonico usando retrieve_lore_for_elements
3. VALIDAR el guion contra el lore usando validate_against_lore
4. EMITIR veredicto con emit_verdict
5. GUARDAR el plan con save_plan

Fragmento a analizar:
""" + script_fragment + """

Instrucciones:
- Sigue las etapas en ORDEN.
- Si detectas una inconsistencia CRITICA en la etapa 3, el veredicto debe ser INCONSISTENTE.
- Si no hay inconsistencias, el veredicto es CONSISTENTE.
- Usa las herramientas, no inventes informacion.
"""

        print(f"\n[EXEC] Iniciando ejecucion del plan...")
        result = self.executor.invoke({"input": orchestration_prompt})

        print(f"\n{'='*70}\n RESULTADO FINAL\n{'='*70}")
        print(result["output"])
        print(f"{'='*70}\n")
        return result

## Demostracion

Tres fragmentos de prueba que demuestran planificacion y toma de decisiones adaptativa.

In [ ]:
agent = DeliberativeContinuityAgent(verbose=True)

print("\n" + "="*70)
print(" DEMO: Agente Deliberativo con Planificacion")
print("="*70)

In [ ]:
agent.run(
    "Escena 12 — 2007. Bumblebee se gira hacia Sam y dice en voz alta: '
    'Sam, debes llevar el AllSpark al Secretario de Defensa.'"
)

In [ ]:
agent.run(
    "Ano 2010. Sam Witwicky y Cade Yeager se reunen en el cuartel de NEST '
    'para analizar fragmentos del AllSpark."
)

In [ ]:
agent.run(
    "Optimus Prime despliega su espada energetica y enfrenta a Megatron '
    'en las calles de Mission City. Sam corre hacia el callejon sosteniendo el AllSpark."
)